[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_79_Routing_and_Handoff.ipynb)

# Lesson 79 — Routing & Handoff
### Phase 9 · Advanced Multi-Agent Orchestration · Lesson 3 of ~6

In **Lesson 77** you learned that a multi-agent system is a *control-flow topology* over LLM calls, and in **Lesson 78** you built a **blackboard** so agents coordinate through shared state instead of O(N²) point-to-point wiring.

Both lessons assumed something we quietly swept under the rug: **you already knew which agent should do what.** Real systems don't get that luxury. A support inbox, an internal copilot, an API gateway — they receive a *heterogeneous stream* of tasks and something has to decide, per task, **who handles this?**

> **Core through-line for today:** A **router** is a cheap classifier that reads each incoming task and *dispatches it to the best specialist*. When it isn't sure, it **escalates** (to a generalist, a bigger model, or a human). When a specialist realizes a task isn't really its job, it performs a **handoff** — a mid-flight transfer to a better-suited agent. Routing decides *up front*; handoff corrects *in flight*. The whole thing is a **load balancer with judgment** — and the router itself becomes your new single point of failure.

| Lesson | Topic | Idea |
|---|---|---|
| L77 | Topologies | wiring sets latency & who-decides-done |
| L78 | Blackboard | coordinate via shared state, not messages |
| **L79 (today)** | **Routing & Handoff** | **classify → dispatch → escalate / hand off** |
| L80 | Planning & Decomposition | an agent that *plans* the sub-tasks itself |
| L81 | Reliability & partial failure | what happens when an agent dies mid-task |
| L82 | Phase-9 capstone | ship a 4th OSS artifact: an orchestration app |

By the end you'll have added **`orchestra/router.py`** to the package you started in L77 (`core.py`) and L78 (`blackboard.py`).

In [ ]:
# --- Setup: no API key, no network, fully deterministic ---
!pip install rich -q

import os, sys, base64, hashlib, importlib
from rich import print as rprint
from rich.table import Table

# In Google Colab this is /content. Everything we write lives under BASE.
BASE = "/content"
os.makedirs(os.path.join(BASE, "orchestra"), exist_ok=True)

# Ship the orchestra package modules to disk (base64 => collision-proof source).
_CORE_B64 = "IyBvcmNoZXN0cmEvY29yZS5weSAgLS0gIG1lc3NhZ2UgKyBhZ2VudCBwcmltaXRpdmVzIChmcm9tIExlc3NvbiA3NykKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZQoKQGRhdGFjbGFzcwpjbGFzcyBNZXNzYWdlOgogICAgc2VuZGVyOiBzdHIKICAgIHJlY2lwaWVudDogc3RyCiAgICBraW5kOiBzdHIgICAgICAgICAgICAgICAgICMgInRhc2siIHwgInJlc3VsdCIgfCAiaGFuZG9mZiIgfCAuLi4KICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKY2xhc3MgQWdlbnQ6CiAgICAjIEFuIGFnZW50ID0gb25lIGNhbGxhYmxlICJicmFpbiIgYmVoaW5kIGEgbmFtZS4gYmFja2VuZChuYW1lLCBjb250ZW50KSAtPiAob3V0cHV0LCB0b2tlbnMpCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCByb2xlOiBzdHIsIGJhY2tlbmQ6IENhbGxhYmxlKToKICAgICAgICBzZWxmLm5hbWUgPSBuYW1lCiAgICAgICAgc2VsZi5yb2xlID0gcm9sZQogICAgICAgIHNlbGYuYmFja2VuZCA9IGJhY2tlbmQKICAgICAgICBzZWxmLmNhbGxzID0gMAogICAgZGVmIGFjdChzZWxmLCBtc2c6ICJNZXNzYWdlIikgLT4gIk1lc3NhZ2UiOgogICAgICAgIHNlbGYuY2FsbHMgKz0gMQogICAgICAgIG91dCwgdG9rZW5zID0gc2VsZi5iYWNrZW5kKHNlbGYubmFtZSwgbXNnLmNvbnRlbnQpCiAgICAgICAgcmV0dXJuIE1lc3NhZ2Uoc2VuZGVyPXNlbGYubmFtZSwgcmVjaXBpZW50PW1zZy5zZW5kZXIsCiAgICAgICAgICAgICAgICAgICAgICAga2luZD0icmVzdWx0IiwgY29udGVudD1vdXQsIG1ldGE9eyJ0b2tlbnMiOiB0b2tlbnN9KQo="
_ROUTER_B64 = "IyBvcmNoZXN0cmEvcm91dGVyLnB5ICAtLSAgYSByb3V0ZXIgdGhhdCBjbGFzc2lmaWVzIGVhY2ggdGFzayBhbmQgaGFuZHMgaXQKIyB0byB0aGUgYmVzdCBzcGVjaWFsaXN0LCBlc2NhbGF0ZXMgd2hlbiB1bnN1cmUsIGFuZCBzdXJ2aXZlcyBoYW5kb2ZmIGxvb3BzLgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgUm91dGU6CiAgICBjYXRlZ29yeTogT3B0aW9uYWxbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKCmNsYXNzIENsYXNzaWZpZXI6CiAgICAjIENoZWFwIGtleXdvcmQgY2xhc3NpZmllci4gSW4gcHJvZHVjdGlvbiB0aGlzIHdvdWxkIGJlIGFuIGVtYmVkZGluZyBtb2RlbAogICAgIyBvciBhIHNtYWxsIExMTTsgdGhlIGludGVyZmFjZSAodGV4dCAtPiBSb3V0ZSkgaXMgd2hhdCBtYXR0ZXJzLgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtleXdvcmRzOiBEaWN0W3N0ciwgbGlzdF0pOgogICAgICAgIHNlbGYua2V5d29yZHMgPSBrZXl3b3JkcwogICAgICAgIHNlbGYuY2FsbHMgPSAwCiAgICBkZWYgY2xhc3NpZnkoc2VsZiwgdGV4dDogc3RyKSAtPiBSb3V0ZToKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICB0ID0gdGV4dC5sb3dlcigpCiAgICAgICAgc2NvcmVzID0ge30KICAgICAgICBmb3IgY2F0LCBrd3MgaW4gc2VsZi5rZXl3b3Jkcy5pdGVtcygpOgogICAgICAgICAgICBoaXRzID0gc3VtKDEgZm9yIGsgaW4ga3dzIGlmIGsgaW4gdCkKICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgIHNjb3Jlc1tjYXRdID0gaGl0cwogICAgICAgIGlmIG5vdCBzY29yZXM6CiAgICAgICAgICAgIHJldHVybiBSb3V0ZShOb25lLCAwLjApICAgICAgICAgICAgICAjIG5vdGhpbmcgbWF0Y2hlZCAtPiBlc2NhbGF0ZQogICAgICAgIHRvdGFsID0gc3VtKHNjb3Jlcy52YWx1ZXMoKSkKICAgICAgICAjIGRldGVybWluaXN0aWMgdGllLWJyZWFrOiBtb3N0IGhpdHMgZmlyc3QsIHRoZW4gY2F0ZWdvcnkgbmFtZQogICAgICAgIGJlc3RfY2F0LCBiZXN0X2hpdHMgPSBzb3J0ZWQoc2NvcmVzLml0ZW1zKCksIGtleT1sYW1iZGEga3Y6ICgta3ZbMV0sIGt2WzBdKSlbMF0KICAgICAgICByZXR1cm4gUm91dGUoYmVzdF9jYXQsIGJlc3RfaGl0cyAvIHRvdGFsKQoKY2xhc3MgUm91dGVyOgogICAgIyBIb2xkcyBzcGVjaWFsaXN0cyAoY2F0ZWdvcnkgLT4gQWdlbnQpLCBhIGdlbmVyYWxpc3QgZmFsbGJhY2ssIGFuZCB0aGUKICAgICMgcG9saWN5OiBnYXRlIG9uIGNvbmZpZGVuY2UsIGRpc3BhdGNoLCBmb2xsb3cgaGFuZG9mZnMsIGNhcCB0aGUgaG9wcy4KICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjbGFzc2lmaWVyLCBzcGVjaWFsaXN0cywgZ2VuZXJhbGlzdCwKICAgICAgICAgICAgICAgICB0aHJlc2hvbGQ9MC42LCBob3BfY2FwPTMsIGNsYXNzaWZ5X2Nvc3Q9Mik6CiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gY2xhc3NpZmllcgogICAgICAgIHNlbGYuc3BlY2lhbGlzdHMgPSBkaWN0KHNwZWNpYWxpc3RzKQogICAgICAgIHNlbGYuZ2VuZXJhbGlzdCA9IGdlbmVyYWxpc3QKICAgICAgICBzZWxmLnRocmVzaG9sZCA9IHRocmVzaG9sZAogICAgICAgIHNlbGYuaG9wX2NhcCA9IGhvcF9jYXAKICAgICAgICBzZWxmLmNsYXNzaWZ5X2Nvc3QgPSBjbGFzc2lmeV9jb3N0CgogICAgZGVmIF9lc2NhbGF0ZShzZWxmLCB0YXNrLCB0cmFjZSwgdG9rZW5zLCByZWFzb24pOgogICAgICAgIGhvcHMgPSBsZW4odHJhY2UpICAgICAgICAgICAgICAgICAgICAgICAgIyBzcGVjaWFsaXN0IGNhbGxzIG1hZGUgYmVmb3JlIGdpdmluZyB1cAogICAgICAgIG0gPSBzZWxmLmdlbmVyYWxpc3QuYWN0KE1lc3NhZ2UoInJvdXRlciIsIHNlbGYuZ2VuZXJhbGlzdC5uYW1lLCAidGFzayIsIHRhc2spKQogICAgICAgIHRyYWNlLmFwcGVuZChzZWxmLmdlbmVyYWxpc3QubmFtZSkKICAgICAgICB0b2tlbnMgKz0gbS5tZXRhLmdldCgidG9rZW5zIiwgMCkKICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywgInRyYWNlIjogdHJhY2UsCiAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6IHJlYXNvbiwgImhvcHMiOiBob3BzfQoKICAgIGRlZiBkaXNwYXRjaChzZWxmLCB0YXNrKToKICAgICAgICB0cmFjZSA9IFtdCiAgICAgICAgdG9rZW5zID0gc2VsZi5jbGFzc2lmeV9jb3N0CiAgICAgICAgcm91dGUgPSBzZWxmLmNsYXNzaWZpZXIuY2xhc3NpZnkodGFza1sidGV4dCJdKQogICAgICAgICMgR0FURTogaWYgdGhlIGNsYXNzaWZpZXIgaXMgdW5zdXJlLCBkb24ndCBndWVzcyBhIHNwZWNpYWxpc3QgLS0gZXNjYWxhdGUuCiAgICAgICAgaWYgcm91dGUuY2F0ZWdvcnkgaXMgTm9uZSBvciByb3V0ZS5jb25maWRlbmNlIDwgc2VsZi50aHJlc2hvbGQ6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lc2NhbGF0ZSh0YXNrLCB0cmFjZSwgdG9rZW5zLCAibG93X2NvbmZpZGVuY2UiKQogICAgICAgIGN1cnJlbnQgPSByb3V0ZS5jYXRlZ29yeQogICAgICAgIGhvcHMgPSAwCiAgICAgICAgd2hpbGUgaG9wcyA8IHNlbGYuaG9wX2NhcDoKICAgICAgICAgICAgaG9wcyArPSAxCiAgICAgICAgICAgIGlmIGN1cnJlbnQgbm90IGluIHNlbGYuc3BlY2lhbGlzdHM6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgInVua25vd25fY2F0ZWdvcnkiKQogICAgICAgICAgICBhZ2VudCA9IHNlbGYuc3BlY2lhbGlzdHNbY3VycmVudF0KICAgICAgICAgICAgbSA9IGFnZW50LmFjdChNZXNzYWdlKCJyb3V0ZXIiLCBhZ2VudC5uYW1lLCAidGFzayIsIHRhc2spKQogICAgICAgICAgICB0cmFjZS5hcHBlbmQoYWdlbnQubmFtZSkKICAgICAgICAgICAgdG9rZW5zICs9IG0ubWV0YS5nZXQoInRva2VucyIsIDApCiAgICAgICAgICAgIHRhcmdldCA9IG0uY29udGVudC5nZXQoImhhbmRvZmYiKQogICAgICAgICAgICBpZiB0YXJnZXQgaXMgTm9uZTogICAgICAgICAgICAgICAgICAgIyBzcGVjaWFsaXN0IG93bmVkIGl0IC0+IGRvbmUKICAgICAgICAgICAgICAgIHJldHVybiB7ImFuc3dlciI6IG0uY29udGVudC5nZXQoImFuc3dlciIpLCAidG9rZW5zIjogdG9rZW5zLAogICAgICAgICAgICAgICAgICAgICAgICAidHJhY2UiOiB0cmFjZSwgImVzY2FsYXRlZCI6IEZhbHNlLCAicmVhc29uIjogInJvdXRlZCIsICJob3BzIjogaG9wc30KICAgICAgICAgICAgY3VycmVudCA9IHRhcmdldCAgICAgICAgICAgICAgICAgICAgICMgSEFORE9GRjogdHJhbnNmZXIgdG8gYW5vdGhlciBzcGVjaWFsaXN0CiAgICAgICAgIyBMT09QIEdVQVJEOiB0b28gbWFueSBoYW5kb2ZmcyAtPiBzdG9wIHRoZSBtZXJyeS1nby1yb3VuZCwgZXNjYWxhdGUuCiAgICAgICAgcmV0dXJuIHNlbGYuX2VzY2FsYXRlKHRhc2ssIHRyYWNlLCB0b2tlbnMsICJob3BfY2FwIikK"
with open(os.path.join(BASE, "orchestra", "core.py"), "w") as f:
    f.write(base64.b64decode(_CORE_B64).decode())
with open(os.path.join(BASE, "orchestra", "router.py"), "w") as f:
    f.write(base64.b64decode(_ROUTER_B64).decode())
# a package __init__ so `from orchestra.router import ...` works on a fresh kernel
open(os.path.join(BASE, "orchestra", "__init__.py"), "w").close()

if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()

from orchestra.core import Message, Agent
from orchestra.router import Route, Classifier, Router

rprint("[green]OK[/] orchestra.core + orchestra.router written and imported. Deterministic, no API key needed.")

## 1. The problem: one big agent is a jack-of-all-trades

The lazy design is **"send everything to one powerful generalist."** It works, but it's the worst of both worlds: you pay premium-model prices on *every* task, and a generalist is, by definition, mediocre at each specialty compared with a focused expert.

Routing flips this: a **cheap classifier** looks at each task and sends it to a **cheap specialist** that is *excellent* at exactly that one thing. You only reach for the expensive generalist when you're genuinely unsure.

| | Generalist-only | Router + specialists |
|---|---|---|
| Cost per task | high (big model, every time) | low (small specialist, usually) |
| Accuracy in-domain | mediocre everywhere | excellent in the right lane |
| Failure mode | uniformly so-so | **mis-routing** (confidently wrong) |
| New capability | retrain / re-prompt the monolith | register one more specialist |

We'll build a tiny **support desk** with four task types — `billing`, `code`, `math`, `translate` — and measure both designs on the *same* ground-truth suite.

In [ ]:
# --- A ground-truth task suite + deterministic specialist/generalist backends ---
# Each task carries `fields` with exactly ONE populated field = its true category.
# A specialist is EXCELLENT on its own field and blind off-domain -- just like a
# real narrow model. The generalist can read any field but is only ~70% reliable.

CATS = ["billing", "code", "math", "translate"]
FIELD_OF = {"billing": "amount", "code": "name", "math": "nums", "translate": "word"}

KEYWORDS = {
    "billing":   ["invoice", "refund", "charge", "payment", "bill"],
    "code":      ["bug", "function", "stacktrace", "compile", "code"],
    "math":      ["compute", "product", "multiply", "times", "calculate"],
    "translate": ["translate", "spanish", "language", "word"],
}

def solve_in_domain(cat, task):
    f = task["fields"]
    if cat == "billing":   return "refunded:%s" % f["amount"]
    if cat == "code":      return "patched:%s" % f["name"]
    if cat == "math":      return str(f["nums"][0] * f["nums"][1])
    if cat == "translate": return "es:%s" % f["word"]

def detect_category(task):
    # A specialist CAN glance at a task and see which field is populated,
    # even if it isn't its own -- that is how a directed handoff picks a target.
    for cat, field in FIELD_OF.items():
        if task["fields"].get(field) is not None:
            return cat
    return None

def make_task(i, cat, text, fields):
    t = {"id": "t%03d" % i, "cat": cat, "text": text,
         "fields": {FIELD_OF[c]: None for c in CATS}}
    t["fields"].update(fields)
    t["gold"] = solve_in_domain(cat, t)
    return t

# 15 CLEAR tasks per category (single-category keywords -> confidence 1.0)
tasks, i = [], 0
for k in range(15):
    tasks.append(make_task(i, "billing",   "please refund the invoice charge",      {"amount": 100 + k}));            i += 1
    tasks.append(make_task(i, "code",      "there is a bug in this function code",   {"name": "fn_%d" % k}));          i += 1
    tasks.append(make_task(i, "math",      "compute the product, multiply the times", {"nums": (k + 2, 3)}));          i += 1
    tasks.append(make_task(i, "translate", "translate this word to spanish",          {"word": "w%d" % k}));           i += 1

# 20 AMBIGUOUS tasks: text carries TWO categories' keywords (confidence ~0.5).
# For 14 of them the distractor sorts first (greedy argmax picks WRONG);
# for 6 the true category sorts first (greedy happens to be right).
AMB = [("math", "billing"), ("code", "billing"), ("translate", "billing"),
       ("math", "code"), ("translate", "code"), ("billing", "code"),
       ("math", "translate")]
amb_fields = {"billing": {"amount": 900}, "code": {"name": "amb"},
              "math": {"nums": (7, 8)}, "translate": {"word": "amb"}}
for k in range(20):
    true_cat, distractor = AMB[k % len(AMB)]
    kw_true = KEYWORDS[true_cat][0]
    kw_dis  = KEYWORDS[distractor][0]
    text = "%s ... %s" % (kw_dis, kw_true)   # one keyword each -> conf 0.5
    tasks.append(make_task(i, true_cat, text, amb_fields[true_cat])); i += 1

clear_tasks = [t for t in tasks if "..." not in t["text"]]
amb_tasks   = [t for t in tasks if "..." in t["text"]]

def specialist_backend(cat, self_check):
    field = FIELD_OF[cat]
    def backend(name, task):
        if task["fields"].get(field) is not None:          # in-domain: nail it, cheap
            return ({"answer": solve_in_domain(cat, task), "handoff": None}, 20)
        if self_check:                                     # off-domain + self-aware: hand off
            return ({"answer": None, "handoff": detect_category(task)}, 8)
        return ({"answer": "%s?:guess" % cat, "handoff": None}, 20)   # off-domain, naive: confidently wrong
    return backend

def generalist_backend(name, task):
    true = detect_category(task)
    h = int(hashlib.md5(task["id"].encode()).hexdigest(), 16) % 10
    if h < 7:                                              # ~70% reliable, expensive
        return ({"answer": solve_in_domain(true, task), "handoff": None}, 60)
    return ({"answer": "general?:guess", "handoff": None}, 60)

generalist = Agent("generalist", "does-everything-okay", generalist_backend)

def is_correct(res, task):
    return res["answer"] == task["gold"]

# self-tests
_b = Agent("billing", "billing", specialist_backend("billing", self_check=False))
_bt = [t for t in clear_tasks if t["cat"] == "billing"][0]
assert is_correct({"answer": _b.act(Message("r","billing","task",_bt)).content["answer"]}, _bt)
_ct = [t for t in clear_tasks if t["cat"] == "code"][0]
assert not is_correct({"answer": _b.act(Message("r","billing","task",_ct)).content["answer"]}, _ct)  # billing blind on code
assert len(tasks) == 80 and len(clear_tasks) == 60 and len(amb_tasks) == 20
rprint("[green]OK[/] 80 tasks (60 clear + 20 ambiguous). Specialists nail their lane, are blind off it.")

In [ ]:
# --- Baseline: send EVERYTHING to the generalist ---
def run_generalist_only(suite):
    correct = tokens = 0
    for t in suite:
        m = generalist.act(Message("router", "generalist", "task", t))
        tokens += m.meta["tokens"]
        if is_correct({"answer": m.content["answer"]}, t):
            correct += 1
    return correct / len(suite), tokens

BASE_ACC, BASE_TOK = run_generalist_only(tasks)
rprint("Generalist-only: [yellow]accuracy %.3f[/], tokens [yellow]%d[/]" % (BASE_ACC, BASE_TOK))
assert 0.60 <= BASE_ACC <= 0.80, BASE_ACC        # decent but not great, and...
assert BASE_TOK == 80 * 60                        # ...we paid premium price on every single task
rprint("[green]OK[/] Baseline is mediocre AND expensive -- one big brain for everything.")

## 2. The classifier: intent → `(category, confidence)`

Routing starts with **classification**: read the task, decide its category, and — crucially — attach a **confidence**. Confidence is what lets the router *know when it doesn't know*.

Our `Classifier` (already in `orchestra/router.py`) counts keyword hits per category. `confidence = hits_for_winner / total_hits`:

- A **clear** task hits one category only → confidence **1.0**.
- An **ambiguous** task hits two categories equally → confidence **0.5**.
- A task matching **nothing** → `(None, 0.0)` → automatic escalate.

In production you'd swap the keyword counter for an embedding-similarity or small-LLM classifier — the `text → Route` interface stays identical.

In [ ]:
clf = Classifier(KEYWORDS)

clear_billing = [t for t in clear_tasks if t["cat"] == "billing"][0]
r_clear = clf.classify(clear_billing["text"])
assert r_clear.category == "billing" and r_clear.confidence == 1.0

r_amb = clf.classify(amb_tasks[0]["text"])
assert 0.0 < r_amb.confidence < 0.6            # ambiguous => low confidence => will escalate

r_none = clf.classify("hello there, nothing relevant here")
assert r_none.category is None and r_none.confidence == 0.0

rprint("clear -> %s  |  ambiguous -> %s  |  no-match -> %s" %
       (r_clear, r_amb, r_none))
rprint("[green]OK[/] Confidence is the signal the router uses to decide when to trust a specialist.")

## 3. The router: dispatch, and escalate when unsure  ← THE PAYOFF

The `Router` policy:

1. Classify the task.
2. **Gate:** if confidence `< threshold` (0.6) or no category → **escalate** to the generalist. Don't guess.
3. Otherwise dispatch to that category's specialist and return its answer.

So clear tasks (conf 1.0) go to cheap experts; ambiguous tasks (conf 0.5) fall back to the generalist. Let's measure the same 80-task suite and compare against the baseline.

In [ ]:
# Naive specialists (self_check=False) so this section is pure routing, no handoff yet.
specialists = {c: Agent(c, c + "-expert", specialist_backend(c, self_check=False)) for c in CATS}
router = Router(clf, specialists, generalist, threshold=0.6)

def run_router(suite, rtr):
    correct = tokens = escalations = 0
    for t in suite:
        res = rtr.dispatch(t)
        tokens += res["tokens"]
        escalations += 1 if res["escalated"] else 0
        if is_correct(res, t):
            correct += 1
    return correct / len(suite), tokens, escalations

ROUTED_ACC, ROUTED_TOK, ESC = run_router(tasks, router)

tbl = Table(title="Generalist-only  vs  Router+specialists")
tbl.add_column("design"); tbl.add_column("accuracy", justify="right"); tbl.add_column("tokens", justify="right")
tbl.add_row("generalist-only", "%.3f" % BASE_ACC, str(BASE_TOK))
tbl.add_row("router (gated)",  "%.3f" % ROUTED_ACC, str(ROUTED_TOK))
rprint(tbl)
rprint("router escalated %d/%d tasks (the ambiguous ones) to the generalist" % (ESC, len(tasks)))

assert ROUTED_ACC > BASE_ACC + 0.15            # specialists nail their lane
assert ROUTED_TOK < BASE_TOK * 0.70            # and we stopped paying premium on every task
assert ESC == 20                               # exactly the 20 ambiguous tasks were escalated
rprint("[green]PAYOFF[/] Higher accuracy AND lower cost -- because most tasks went to a cheap expert, "
       "and only the genuinely-unsure ones cost a generalist call.")

## 4. The router is your new single point of failure

Routing concentrates risk. If the classifier sends a task to the **wrong** specialist, that specialist is *blind off-domain* and will answer **confidently wrong** — worse than a generalist's honest 70%.

Let's simulate a classifier that mislabels **40% of clear tasks** with high confidence, and watch accuracy fall through the floor.

In [ ]:
class NoisyClassifier(Classifier):
    # Deterministically corrupts ~40% of confident classifications.
    def classify(self, text):
        r = super().classify(text)
        if r.category is not None and r.confidence >= 0.6:
            h = int(hashlib.md5(text.encode()).hexdigest(), 16) % 10
            if h < 4:                                  # 40% get a WRONG label, still confident
                wrong = sorted(c for c in CATS if c != r.category)[h % 3]
                return Route(wrong, 0.95)
        return r

noisy_router = Router(NoisyClassifier(KEYWORDS), specialists, generalist, threshold=0.6)
NOISY_ACC, _, _ = run_router(tasks, noisy_router)
rprint("router with a 40%%-noisy classifier: accuracy [red]%.3f[/] (was %.3f)" % (NOISY_ACC, ROUTED_ACC))
assert NOISY_ACC < ROUTED_ACC - 0.15           # mis-routing => confidently wrong answers

# --- Why the confidence GATE matters: greedy (always argmax) vs gated (escalate if unsure) ---
def run_on(subset, threshold):
    rtr = Router(clf, specialists, generalist, threshold=threshold)
    acc, _, _ = run_router(subset, rtr)
    return acc

greedy_acc = run_on(amb_tasks, threshold=0.0)   # never escalate: trust the coin-flip argmax
gated_acc  = run_on(amb_tasks, threshold=0.6)   # escalate the unsure ones to the generalist
rprint("ambiguous subset -- greedy(argmax): [red]%.3f[/]   gated(escalate): [green]%.3f[/]"
       % (greedy_acc, gated_acc))
assert gated_acc > greedy_acc                   # knowing when NOT to trust a specialist is the whole point
rprint("[green]OK[/] A confident wrong route is the dangerous failure. The gate converts "
       "'I'm not sure' into an escalation instead of a confident mistake.")

## 5. Handoff: correcting a bad route *in flight*

Escalation is one safety net (kick unsure tasks up to a generalist). **Handoff** is the other: a specialist that receives a task, realizes *"this isn't mine,"* and **transfers it to a better-suited specialist** — carrying the task with it.

- **Routing** is the *up-front* decision (router → specialist).
- **Handoff** is a *peer-to-peer* correction (specialist → specialist), mid-flight.

Our self-aware specialists (`self_check=True`) look at the task, see their own field is empty, and hand off to the specialist whose field *is* populated. The `Router` follows the handoff chain — but every transfer costs a hop, and two stubborn agents could ping-pong forever. So we **cap the hops** and escalate if the cap is hit.

In [ ]:
# Self-aware specialists that hand off instead of guessing.
smart = {c: Agent(c, c + "-expert", specialist_backend(c, self_check=True)) for c in CATS}

# (a) A single task deliberately mis-routed: force 'billing' on a real CODE task.
code_task = [t for t in clear_tasks if t["cat"] == "code"][0]
class ForceBilling(Classifier):
    def classify(self, text): return Route("billing", 1.0)

no_handoff = Router(ForceBilling(KEYWORDS),
                    {c: Agent(c, c, specialist_backend(c, self_check=False)) for c in CATS},
                    generalist, threshold=0.6, hop_cap=3)
with_handoff = Router(ForceBilling(KEYWORDS), smart, generalist, threshold=0.6, hop_cap=3)

res_bad  = no_handoff.dispatch(code_task)
res_good = with_handoff.dispatch(code_task)
rprint("mis-routed to billing --  no handoff: answer=%r correct=%s" %
       (res_bad["answer"], is_correct(res_bad, code_task)))
rprint("mis-routed to billing -- with handoff: trace=%s answer=%r correct=%s" %
       (res_good["trace"], res_good["answer"], is_correct(res_good, code_task)))
assert not is_correct(res_bad, code_task)                         # blind billing agent guessed wrong
assert is_correct(res_good, code_task)                            # handoff rescued it
assert res_good["trace"] == ["billing", "code"] and res_good["hops"] == 2

# (b) Whole-suite recovery: same 40%-noisy classifier, but now with self-aware specialists.
handoff_router = Router(NoisyClassifier(KEYWORDS), smart, generalist, threshold=0.6)
HANDOFF_ACC, _, _ = run_router(tasks, handoff_router)
rprint("noisy classifier + handoff: accuracy [green]%.3f[/] (noisy without handoff was %.3f)"
       % (HANDOFF_ACC, NOISY_ACC))
assert HANDOFF_ACC > NOISY_ACC + 0.20                             # handoff repairs most mis-routes

# (c) LOOP GUARD: two agents that always hand off to each other.
def stubborn(target):
    def backend(name, task): return ({"answer": None, "handoff": target}, 5)
    return backend
ping = Agent("ping", "loops", stubborn("pong"))
pong = Agent("pong", "loops", stubborn("ping"))
loop_clf = Classifier({"ping": ["ping"], "pong": ["pong"]})
loop_router = Router(loop_clf, {"ping": ping, "pong": pong}, generalist, threshold=0.6, hop_cap=3)
loop_res = loop_router.dispatch({"id": "loop", "text": "ping", "fields": {}, "cat": "ping", "gold": None})
rprint("handoff loop -> escalated=%s reason=%r hops=%d" %
       (loop_res["escalated"], loop_res["reason"], loop_res["hops"]))
assert loop_res["escalated"] and loop_res["reason"] == "hop_cap" and loop_res["hops"] == 3
rprint("[green]OK[/] Handoff repairs bad routes; the hop cap stops a ping-pong loop from running forever.")

## 6. Ship it: `orchestra/router.py` is already a real module

We wrote `orchestra/router.py` at the top of the notebook and have been importing `Route`, `Classifier`, and `Router` from it the whole time — so the package now has **three** modules (`core.py` L77, `blackboard.py` L78, `router.py` L79). Let's prove it stands alone with a fresh smoke test.

In [ ]:
import importlib, orchestra.router as R
importlib.reload(R)

# smoke test 1: classifier confidence math
_c = R.Classifier({"a": ["alpha"], "b": ["beta"]})
assert _c.classify("alpha beta").confidence == 0.5
assert _c.classify("nothing").category is None

# smoke test 2: gate escalates a no-match task
_g = Agent("gen", "gen", lambda n, t: ({"answer": "G", "handoff": None}, 1))
_r = R.Router(R.Classifier({"a": ["alpha"]}), {}, _g, threshold=0.6)
_out = _r.dispatch({"text": "zzz", "fields": {}})
assert _out["escalated"] and _out["reason"] == "low_confidence"

# smoke test 3: a clean route with no handoff
_sp = {"a": Agent("a", "a", lambda n, t: ({"answer": "A", "handoff": None}, 2))}
_r2 = R.Router(R.Classifier({"a": ["alpha"]}), _sp, _g, threshold=0.6)
_out2 = _r2.dispatch({"text": "alpha", "fields": {}})
assert _out2["answer"] == "A" and not _out2["escalated"] and _out2["hops"] == 1

rprint("[green]OK[/] orchestra/router.py imports clean and passes 3 independent smoke tests.")

## 7. Ten ways routing goes wrong

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | **No confidence, only argmax** | you can't escalate what you can't measure being unsure about |
| 2 | **Threshold too low** | everything routes greedily → confident mis-routes |
| 3 | **Threshold too high** | everything escalates → you're back to generalist-only cost |
| 4 | **Specialist confidently wrong off-domain** | worse than an honest "I don't know"; add a self-check |
| 5 | **No hop cap on handoff** | two agents ping-pong forever, burning tokens |
| 6 | **Context lost across handoff** | the receiving agent gets the task but not what's been tried |
| 7 | **Classifier drift** | keyword/embedding classifier rots as task distribution shifts |
| 8 | **No fallback category** | a task matching nothing crashes instead of escalating |
| 9 | **Router as bottleneck** | one classifier in front of everything = one thing to overload |
| 10 | **No logging of routes** | you can't debug mis-routes you never recorded (wire in L72 tracing) |

In [ ]:
# --- Verification checklist: every claim this lesson made, re-asserted ---
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))

check("baseline mediocre (0.6-0.8)",        0.60 <= BASE_ACC <= 0.80)
check("baseline expensive (premium/task)",  BASE_TOK == 80 * 60)
check("routed beats baseline by >0.15",     ROUTED_ACC > BASE_ACC + 0.15)
check("routed cheaper (<70% baseline tok)", ROUTED_TOK < BASE_TOK * 0.70)
check("router escalated the 20 ambiguous",  ESC == 20)
check("clear task -> confidence 1.0",       clf.classify(clear_billing["text"]).confidence == 1.0)
check("ambiguous -> confidence < 0.6",      clf.classify(amb_tasks[0]["text"]).confidence < 0.6)
check("no-match -> escalate signal",        clf.classify("irrelevant").category is None)
check("noisy classifier tanks accuracy",    NOISY_ACC < ROUTED_ACC - 0.15)
check("gated beats greedy on ambiguous",    gated_acc > greedy_acc)
check("mis-route without handoff is wrong", not is_correct(res_bad, code_task))
check("handoff rescues the mis-route",      is_correct(res_good, code_task))
check("handoff trace billing->code, 2 hops",res_good["trace"] == ["billing", "code"] and res_good["hops"] == 2)
check("handoff lifts noisy-suite by >0.20", HANDOFF_ACC > NOISY_ACC + 0.20)
check("hop cap escalates a handoff loop",   loop_res["escalated"] and loop_res["reason"] == "hop_cap")

tbl = Table(title="Lesson 79 verification")
tbl.add_column("check"); tbl.add_column("result", justify="right")
passed = 0
for name, ok in checks:
    tbl.add_row(name, "[green]PASS[/]" if ok else "[red]FAIL[/]")
    passed += 1 if ok else 0
rprint(tbl)
assert passed == len(checks), "%d/%d checks passed" % (passed, len(checks))
rprint("[bold green]ALL %d CHECKS PASSED[/]" % len(checks))

## 8. Summary, homework & what's next

| Concept | One-liner |
|---|---|
| **Router** | a cheap classifier that dispatches each task to the best specialist |
| **Confidence gate** | escalate to a generalist when the classifier isn't sure — don't guess |
| **Mis-routing** | the router's own failure mode: a confident wrong answer, worse than an honest generalist |
| **Handoff** | a specialist transfers a task peer-to-peer when it realizes it isn't the right owner |
| **Hop cap** | bounds handoff chains so two agents can't ping-pong forever |
| **Escalation ladder** | specialist → generalist → (in prod) human, each rung more capable & costly |

**The result:** the router turned a mediocre-and-expensive generalist into a system that is **more accurate and cheaper**, and two safety nets — the confidence gate and handoff-with-cap — keep a single bad route from becoming a confident wrong answer.

### 🎯 Homework
1. **Real classifier.** Replace `Classifier` with one that embeds the task text (or calls a small LLM) and routes by cosine similarity to category prototypes. Keep the `text → Route` interface identical.
2. **Cost-aware routing.** Add a per-specialist cost and make the gate consider *expected value*: escalate only when the confidence-weighted risk of a mis-route exceeds the generalist's cost.
3. **Carry context across handoff.** Extend the handoff payload so the receiving specialist sees *what the previous one tried* (append to `task["history"]`), and assert it never repeats a failed attempt.
4. **Trace every route.** Emit one span per routing decision using your L72 tracing module, then query "which category is mis-routed most often?"
5. **Bridge to L80.** Give the router a task that needs *two* specialists in sequence (translate **then** bill). Notice routing alone can't express "do A, then B" — that ordered decomposition is exactly what **planning** solves.

### ⏭️ Next: Lesson 80 — Planning & Decomposition
So far a task goes to *one* agent (maybe via a handoff). But real work is often *"first do X, then Y, then combine."* Next we build an agent that **plans**: it decomposes a goal into an ordered set of sub-tasks, dispatches each (using today's router!), and assembles the result — the step from *"who does this task"* to *"what are the tasks."*

*You now own `orchestra/` with three modules — `core.py`, `blackboard.py`, `router.py` — heading toward the Phase-9 capstone (L82).*